# Shared GRNO-3D for Keller--Segel and Darcy--Boussinesq convection

This notebook is the rank-matched 3D realization of the original GRNO design. Both
volumetric PDEs use the same trainable encoder, decoder, width, depths, axial
ConvNeXt blocks, direct centered--routed fusion, and residual head.

The only task-dependent components are parameter-free:

- the carrier diagnostic;
- the domain boundary metadata;
- the context field (zero or depth coordinate);
- the fixed output projection.

The 3D model is not required to share parameter tensors with GRNO-2D. It implements
the same operator blueprint using native 3D pooling, convolution, and trilinear
sampling.

## Public release: setup before execution

See [notebooks/README.md](README.md) for environment, configuration, dataset requirements, and output interpretation.

- The original code cells and hyperparameters are preserved; stored outputs and execution counts have been cleared.
- Configure `PROFILE`, `TASKS_TO_RUN`, `SEEDS`, and `DATA_ROOT_OVERRIDES` in the first code cell before executing subsequent sections in order. The shipped defaults are `paper`, all supported tasks, and seed `0`, with training and testing enabled.
- Canonical datasets, metadata, and normalization files are required. This notebook loads trajectories; it does not generate or download them. See [data/README.md](../data/README.md).
- `paper` and `pilot` require CUDA. `smoke` can run on CPU with valid data and uses a smaller model and shorter protocol.
- Check the printed `ROOT` and `device`. Set `ROOT` explicitly in the first code cell if automatic project discovery does not match your checkout.
- Each task trains its own model with a common architecture. These shared-core notebooks are an initial code release; the manuscript's experiment-specific protocols and baseline results require their corresponding experiments.

**GPU selection:** the first code cell retains `os.environ['CUDA_VISIBLE_DEVICES'] = '1'`. Adjust or remove this assignment before importing PyTorch to match your GPU or scheduler allocation. Restart the kernel after changing it.


## 1. Imports, paths, and frozen protocol

The shipped default trains both tasks with seed `0` in the `paper` profile.
Select a single task for an initial setup check. Use `SEEDS = (0, 1, 2)` for three
independent seeds.


In [ ]:
from __future__ import annotations

import contextlib
import hashlib
import json
import math
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        if (parent / "data" / "keller_segel3d").is_dir() or \
           (parent / "data" / "porous_convection3d").is_dir():
            return parent
        candidate = parent / "model" / "GRNO"
        if candidate.is_dir():
            return candidate.resolve()
    return cwd


ROOT = locate_project_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def short_hash(payload):
    raw = json.dumps(payload, sort_keys=True, default=str).encode()
    return hashlib.sha256(raw).hexdigest()[:12]


def atomic_torch_save(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".partial")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def safe_torch_load(path, map_location=device):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def capture_rng_state():
    return {
        "python": random.getstate(), "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(torch.as_tensor(state["torch_cpu"], dtype=torch.uint8, device="cpu"))
    if torch.cuda.is_available() and state.get("torch_cuda") is not None:
        torch.cuda.set_rng_state_all([
            torch.as_tensor(value, dtype=torch.uint8, device="cpu")
            for value in state["torch_cuda"]
        ])


def amp_context(enabled=True):
    if enabled and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()


PROFILE = "paper"                    # "smoke", "pilot", or "paper"
ALL_3D_TASKS = ("keller_segel3d", "porous_convection3d")
TASKS_TO_RUN = ALL_3D_TASKS#("keller_segel3d",)   # final paper: ALL_3D_TASKS
SEEDS = (0,)                          # final paper: (0, 1, 2)
DATA_ROOT_OVERRIDES = {}
RUN_TRAINING = True
RUN_TESTING = True
RUN_ROUTE_INTERVENTIONS = True
RESUME_IF_AVAILABLE = True
MODEL_VERSION = "shared-grno3d-direct-route-v1"


@dataclass(frozen=True)
class RunConfig:
    profile: str
    epochs: int
    samples_per_epoch: int
    micro_batch: int
    grad_accum: int
    train_rollout: int
    val_rollout: int
    test_rollout: int
    width: int
    depths: tuple
    lr: float
    weight_decay: float
    grad_weight: float
    spectral_weight: float
    rollout_decay: float = 0.8
    use_amp: bool = True

    @property
    def updates_per_epoch(self):
        return self.samples_per_epoch // self.micro_batch // self.grad_accum


if PROFILE == "paper":
    # One protocol for both volumetric systems.
    RUN = RunConfig(PROFILE, 100, 512, 1, 4, 4, 10, 40,
                    24, (2, 2, 3, 4), 3e-4, 1e-4, 0.05, 0.02)
elif PROFILE == "pilot":
    RUN = RunConfig(PROFILE, 20, 128, 1, 4, 4, 10, 40,
                    24, (2, 2, 3, 4), 3e-4, 1e-4, 0.05, 0.02)
elif PROFILE == "smoke":
    RUN = RunConfig(PROFILE, 2, 4, 1, 2, 2, 3, 5,
                    8, (1, 1, 1, 1), 1e-3, 1e-4, 0.03, 0.0,
                    use_amp=False)
else:
    raise ValueError(PROFILE)

if PROFILE != "smoke" and device.type != "cuda":
    raise RuntimeError("Pilot/paper mode requires CUDA; use PROFILE='smoke' on CPU.")
if not set(TASKS_TO_RUN).issubset(ALL_3D_TASKS):
    raise ValueError(TASKS_TO_RUN)
if RUN.samples_per_epoch % (RUN.micro_batch * RUN.grad_accum):
    raise ValueError("samples_per_epoch must divide the effective batch")

OUTPUT_ROOT = ROOT / "outputs" / "shared_grno3d" / PROFILE / MODEL_VERSION
MANIFEST_ROOT = ROOT / "manifests" / "shared_grno3d_K4_v1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "root": str(ROOT), "device": str(device), "profile": PROFILE,
    "tasks": TASKS_TO_RUN, "seeds": SEEDS, "run": asdict(RUN),
}, indent=2))

## 2. Task specifications and canonical trajectory stores

In [ ]:
@dataclass(frozen=True)
class TaskSpec3D:
    name: str
    frame_dt: float
    lengths_zyx: tuple
    boundary_zyx: tuple
    route_kind: str
    condition_kind: str
    conserve_mean: bool
    screening: float = 0.0
    chemotaxis: float = 0.0


TASK_SPECS = {
    "keller_segel3d": TaskSpec3D(
        "keller_segel3d", 0.25, (2 * math.pi, 2 * math.pi, 2 * math.pi),
        ("periodic", "periodic", "periodic"),
        "screened_keller", "zero", True,
        screening=1.0, chemotaxis=0.30),
    "porous_convection3d": TaskSpec3D(
        "porous_convection3d", 0.10, (1.0, 1.0, 1.0),
        ("bounded", "periodic", "periodic"),
        "slab_darcy", "depth", False),
}


def _single_directory(candidates, description):
    candidates = sorted({Path(path).resolve() for path in candidates if Path(path).is_dir()})
    if not candidates:
        raise FileNotFoundError(
            f"No canonical dataset found for {description}. Run the original data-generation notebook first.")
    if len(candidates) > 1:
        raise RuntimeError(
            f"Ambiguous datasets for {description}; set DATA_ROOT_OVERRIDES explicitly:\n" +
            "\n".join(map(str, candidates)))
    return candidates[0]


def discover_data_root(task_name):
    if task_name in DATA_ROOT_OVERRIDES:
        path = Path(DATA_ROOT_OVERRIDES[task_name]).expanduser().resolve()
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    if task_name == "keller_segel3d":
        candidates = [ROOT / "data" / "keller_segel3d" / "paper"]
        candidates += list(ROOT.glob("**/data/keller_segel3d/paper"))
        return _single_directory(candidates, task_name)
    if task_name == "porous_convection3d":
        manifests = list((ROOT / "data" / "porous_convection3d" / "paper").glob(
            "*/dataset_manifest.json"))
        manifests += list(ROOT.glob(
            "**/data/porous_convection3d/paper/*/dataset_manifest.json"))
        valid = []
        for manifest_path in sorted(set(manifests)):
            try:
                payload = json.loads(manifest_path.read_text())
                pde = payload.get("pde", {})
                if (payload.get("dataset_version") ==
                        "porous-convection3d-fv-strang-rk2-continuum-ic-v2"
                        and int(pde.get("model_n", -1)) == 64
                        and float(pde.get("rayleigh_darcy", -1)) == 1000.0):
                    valid.append(manifest_path.parent)
            except Exception:
                pass
        return _single_directory(valid, task_name)
    raise KeyError(task_name)


DATA_ROOTS = {name: discover_data_root(name) for name in TASKS_TO_RUN}


class TrajectoryStore3D:
    def __init__(self, task_name, split):
        self.task_name = task_name
        self.spec = TASK_SPECS[task_name]
        self.root = DATA_ROOTS[task_name]
        self.split = split
        self.files = sorted((self.root / split).glob("trajectory_*.npy"))
        self.files = [path for path in self.files if "diagnostic" not in path.name]
        if not self.files:
            raise FileNotFoundError(self.root / split)
        first = np.load(self.files[0], mmap_mode="r")
        self.n_trajectories = len(self.files)
        self.n_frames = int(first.shape[0])
        self.spatial_shape = tuple(first.shape[-3:])

    def read_frame(self, trajectory, frame):
        value = np.asarray(np.load(self.files[int(trajectory)], mmap_mode="r")[int(frame)],
                           dtype=np.float32)
        if value.ndim == 3:
            value = value[None]
        if value.shape[0] != 1:
            raise ValueError((self.task_name, value.shape))
        return value

    def read_window(self, trajectory, start, horizon):
        value = np.asarray(np.load(self.files[int(trajectory)], mmap_mode="r")[
            int(start):int(start) + int(horizon) + 1], dtype=np.float32)
        if value.ndim == 4:
            value = value[:, None]
        if value.shape[1] != 1:
            raise ValueError((self.task_name, value.shape))
        return value

    def read_condition(self, trajectory):
        d, h, w = self.spatial_shape
        if self.spec.condition_kind == "zero":
            return np.zeros((1, d, h, w), dtype=np.float32)
        z = ((np.arange(d, dtype=np.float32) + 0.5) / d * 2.0 - 1.0)
        return np.broadcast_to(z[None, :, None, None], (1, d, h, w)).copy()

    @property
    def max_train_start(self):
        return self.n_frames - RUN.train_rollout - 1

    def schema(self):
        sample = np.load(self.files[0], mmap_mode="r")
        return {
            "task": self.task_name, "split": self.split,
            "trajectory_count": self.n_trajectories,
            "trajectory_shape": list(sample.shape), "dtype": str(sample.dtype),
            "root": str(self.root),
        }


STORES = {
    name: {split: TrajectoryStore3D(name, split) for split in ("train", "val", "test")}
    for name in TASKS_TO_RUN
}


def validate_data_contract(task_name):
    root = DATA_ROOTS[task_name]
    if task_name == "keller_segel3d":
        path = root / "metadata.json"
        payload = json.loads(path.read_text())
        if payload.get("dataset_version") != "screened-keller-segel3d-etd2-v1":
            raise RuntimeError(payload.get("dataset_version"))
        pde = payload.get("pde", {})
        expected = {"n": 64, "length": 2 * math.pi, "frame_dt": 0.25,
                    "screening": 1.0, "chemotaxis": 0.30,
                    "substeps_per_frame": 8}
    else:
        path = root / "dataset_manifest.json"
        payload = json.loads(path.read_text())
        if payload.get("dataset_version") != "porous-convection3d-fv-strang-rk2-continuum-ic-v2":
            raise RuntimeError(payload.get("dataset_version"))
        pde = payload.get("pde", {})
        expected = {"model_n": 64, "reference_n": 128,
                    "rayleigh_darcy": 1000.0, "frame_dt": 0.1}
    for key, value in expected.items():
        actual = pde.get(key)
        if isinstance(value, float):
            if not math.isclose(float(actual), value, rel_tol=1e-9, abs_tol=1e-12):
                raise RuntimeError(f"{task_name}.{key}: expected {value}, found {actual}")
        elif actual != value:
            raise RuntimeError(f"{task_name}.{key}: expected {value}, found {actual}")
    return short_hash(payload)


DATA_CONTRACT_HASHES = {name: validate_data_contract(name) for name in TASKS_TO_RUN}
for name in TASKS_TO_RUN:
    print(name, DATA_ROOTS[name])
    for split in ("train", "val", "test"):
        print(" ", STORES[name][split].schema())

## 3. Boundary-aware carrier diagnostics and trilinear routing

In [ ]:
_GRID_CACHE_3D = {}
_GREEN_CACHE_3D = {}


def index_grid3d(d, h, w, device):
    key = (d, h, w, str(device))
    if key not in _GRID_CACHE_3D:
        _GRID_CACHE_3D[key] = torch.meshgrid(
            torch.arange(d, device=device, dtype=torch.float32),
            torch.arange(h, device=device, dtype=torch.float32),
            torch.arange(w, device=device, dtype=torch.float32), indexing="ij")
    return _GRID_CACHE_3D[key]


def boundary_warp3d(field, displacement, lengths_zyx, boundary_zyx):
    """Read field at x+d; displacement channels are (dx,dy,dz)."""
    if field.ndim != 5 or displacement.shape != (field.shape[0], 3, *field.shape[-3:]):
        raise ValueError((field.shape, displacement.shape))
    lz, ly, lx = map(float, lengths_zyx)
    bz, by, bx = boundary_zyx
    b, _, d, h, w = field.shape
    z, y, x = index_grid3d(d, h, w, field.device)
    disp = displacement.float()
    sx = x[None] + disp[:, 0] / (lx / w)
    sy = y[None] + disp[:, 1] / (ly / h)
    sz = z[None] + disp[:, 2] / (lz / d)
    sx = torch.remainder(sx, float(w)) if bx == "periodic" else sx.clamp(0, w - 1)
    sy = torch.remainder(sy, float(h)) if by == "periodic" else sy.clamp(0, h - 1)
    sz = torch.remainder(sz, float(d)) if bz == "periodic" else sz.clamp(0, d - 1)

    padded = field.float()
    px = bx == "periodic"
    py = by == "periodic"
    pz = bz == "periodic"
    if px:
        padded = F.pad(padded, (1, 1, 0, 0, 0, 0), mode="circular")
    if py:
        padded = F.pad(padded, (0, 0, 1, 1, 0, 0), mode="circular")
    if pz:
        padded = F.pad(padded, (0, 0, 0, 0, 1, 1), mode="circular")
    sx, sy, sz = sx + int(px), sy + int(py), sz + int(pz)
    dp, hp, wp = padded.shape[-3:]
    gx = 2 * sx / max(wp - 1, 1) - 1
    gy = 2 * sy / max(hp - 1, 1) - 1
    gz = 2 * sz / max(dp - 1, 1) - 1
    grid = torch.stack((gx, gy, gz), dim=-1)
    return F.grid_sample(
        padded, grid, mode="bilinear", padding_mode="border",
        align_corners=True).to(field.dtype)


def midpoint_departure_3d(carrier, dt, spec):
    midpoint = boundary_warp3d(
        carrier, -0.5 * float(dt) * carrier,
        spec.lengths_zyx, spec.boundary_zyx)
    return -float(dt) * midpoint


def dct2_complex(field):
    n = field.shape[1]
    extended = torch.cat((field, torch.flip(field, dims=(1,))), dim=1)
    spectrum = torch.fft.fft(extended, dim=1)
    k = torch.arange(n, device=field.device, dtype=torch.float32)
    phase = torch.exp(-0.5j * math.pi * k / n).reshape(1, n, 1, 1)
    return 0.5 * spectrum[:, :n] * phase


def idct2_complex(coefficients):
    n = coefficients.shape[1]
    k = torch.arange(n, device=coefficients.device, dtype=torch.float32)
    phase = torch.exp(0.5j * math.pi * k / n).reshape(1, n, 1, 1)
    spectrum = coefficients.new_zeros(
        coefficients.shape[0], 2 * n,
        coefficients.shape[2], coefficients.shape[3])
    spectrum[:, :n] = 2.0 * coefficients * phase
    if n > 1:
        reverse_k = torch.arange(n - 1, 0, -1, device=coefficients.device,
                                 dtype=torch.float32)
        reverse_phase = torch.exp(-0.5j * math.pi * reverse_k / n).reshape(
            1, n - 1, 1, 1)
        spectrum[:, n + 1:] = (
            2.0 * torch.flip(coefficients[:, 1:], dims=(1,)) * reverse_phase)
    return torch.fft.ifft(spectrum, dim=1)[:, :n]


class SlabDarcyGreen3D(nn.Module):
    """Zero-parameter mixed-boundary Darcy/Leray map C -> (ux,uy,uz)."""
    def __init__(self, n, lengths_zyx):
        super().__init__()
        self.n = int(n)
        depth, ly, lx = map(float, lengths_zyx)
        self.dz, self.dy, self.dx = depth / n, ly / n, lx / n
        iz = torch.arange(n)
        lambda_z = -4 * torch.sin(math.pi * iz / (2 * n)).square() / self.dz ** 2
        iy = torch.fft.fftfreq(n) * n
        ix = torch.arange(n // 2 + 1)
        lambda_y = -4 * torch.sin(math.pi * iy / n).square() / self.dy ** 2
        lambda_x = -4 * torch.sin(math.pi * ix / n).square() / self.dx ** 2
        denominator = lambda_z[:, None, None] + lambda_y[None, :, None] + lambda_x[None, None, :]
        inverse = torch.where(denominator.abs() > 1e-10,
                              1.0 / denominator, torch.zeros_like(denominator))
        self.register_buffer("poisson_inverse", inverse)

    def _pressure(self, rhs):
        rhs_hat = torch.fft.rfft2(rhs.float(), dim=(-2, -1))
        pressure_hat = idct2_complex(dct2_complex(rhs_hat) * self.poisson_inverse)
        return torch.fft.irfft2(pressure_hat, s=(self.n, self.n), dim=(-2, -1)).real

    def forward(self, concentration):
        q = concentration[:, 0].float()
        b, d, h, w = q.shape
        if (d, h, w) != (self.n, self.n, self.n):
            raise ValueError((q.shape, self.n))
        buoyancy_face = q.new_zeros(b, d + 1, h, w)
        buoyancy_face[:, 1:d] = 0.5 * (q[:, :-1] + q[:, 1:])
        rhs = (buoyancy_face[:, 1:] - buoyancy_face[:, :-1]) / self.dz
        rhs = rhs - rhs.mean((-3, -2, -1), keepdim=True)
        pressure = self._pressure(rhs)
        ux_face = -(torch.roll(pressure, -1, -1) - pressure) / self.dx
        uy_face = -(torch.roll(pressure, -1, -2) - pressure) / self.dy
        grad_z = q.new_zeros(b, d + 1, h, w)
        grad_z[:, 1:d] = (pressure[:, 1:] - pressure[:, :-1]) / self.dz
        uz_face = buoyancy_face - grad_z
        uz_face[:, 0].zero_()
        uz_face[:, -1].zero_()
        ux = 0.5 * (ux_face + torch.roll(ux_face, 1, -1))
        uy = 0.5 * (uy_face + torch.roll(uy_face, 1, -2))
        uz = 0.5 * (uz_face[:, :-1] + uz_face[:, 1:])
        return torch.stack((ux, uy, uz), dim=1)


def keller_carrier(state, spec):
    rho = state[:, 0].float()
    d, h, w = rho.shape[-3:]
    lz, ly, lx = spec.lengths_zyx
    kz = 2 * math.pi * torch.fft.fftfreq(d, d=lz / d, device=rho.device)[:, None, None]
    ky = 2 * math.pi * torch.fft.fftfreq(h, d=ly / h, device=rho.device)[None, :, None]
    kx = 2 * math.pi * torch.fft.fftfreq(w, d=lx / w, device=rho.device)[None, None, :]
    k2 = kx.square() + ky.square() + kz.square()
    rh = torch.fft.fftn(rho, dim=(-3, -2, -1))
    potential = rh / (k2 + spec.screening ** 2)
    ux = torch.fft.ifftn(1j * kx * potential, dim=(-3, -2, -1)).real
    uy = torch.fft.ifftn(1j * ky * potential, dim=(-3, -2, -1)).real
    uz = torch.fft.ifftn(1j * kz * potential, dim=(-3, -2, -1)).real
    return spec.chemotaxis * torch.stack((ux, uy, uz), dim=1)


def raw_carrier_3d(state, spec):
    if spec.route_kind == "screened_keller":
        return keller_carrier(state, spec)
    if spec.route_kind == "slab_darcy":
        key = (state.shape[-1], tuple(spec.lengths_zyx), str(state.device))
        if key not in _GREEN_CACHE_3D:
            _GREEN_CACHE_3D[key] = SlabDarcyGreen3D(
                state.shape[-1], spec.lengths_zyx).to(state.device)
        return _GREEN_CACHE_3D[key](state)
    raise ValueError(spec.route_kind)


def intervene_route_3d(departure, mode):
    if mode == "correct":
        return departure
    if mode == "zero":
        return torch.zeros_like(departure)
    if mode == "reverse":
        return -departure
    if mode == "shuffle":
        return torch.roll(
            departure,
            shifts=(max(departure.shape[-3] // 5, 1),
                    max(departure.shape[-2] // 4, 1),
                    max(departure.shape[-1] // 3, 1)),
            dims=(-3, -2, -1))
    raise ValueError(mode)


def sampler_preflight_3d():
    n, length = 16, 2 * math.pi
    z, y, x = index_grid3d(n, n, n, device)
    field = (torch.sin(2 * math.pi * x / n) + 0.2 * torch.cos(2 * math.pi * z / n))[None, None]
    displacement = torch.zeros(1, 3, n, n, n, device=device)
    displacement[:, 0] = 3 * length / n
    routed = boundary_warp3d(
        field, displacement, (length, length, length),
        ("periodic", "periodic", "periodic"))
    error = float((routed - torch.roll(field, -3, -1)).abs().max())
    identity = boundary_warp3d(
        field, torch.zeros_like(displacement), (1.0, length, length),
        ("bounded", "periodic", "periodic"))
    identity_error = float((identity - field).abs().max())
    print({"periodic_integer_shift_max": error,
           "mixed_identity_max": identity_error})
    assert error < 2e-5 and identity_error < 2e-5


sampler_preflight_3d()

## 4. Frozen preprocessing statistics

State and residual statistics are loaded from the original dataset files.
Component-wise carrier RMS is estimated from a small deterministic subset of
training frames and kept in `STATS` and checkpoint metadata.


In [ ]:
def _scalar(value):
    return float(np.asarray(value).reshape(-1)[0])


def load_base_stats(task_name):
    root = DATA_ROOTS[task_name]
    if task_name == "keller_segel3d":
        paths = sorted(root.glob("training_stats_*.json"))
        if len(paths) != 1:
            raise RuntimeError(f"Expected one Keller stats file, found {paths}")
        raw = json.loads(paths[0].read_text())
        return {
            "state_mean": _scalar(raw.get("state_mean", 0.0)),
            "state_std": _scalar(raw["state_std"]),
            "delta_std": _scalar(raw["delta_std"]),
            "source": str(paths[0]),
        }
    path = root / "train_statistics.json"
    raw = json.loads(path.read_text())
    return {
        "state_mean": _scalar(raw.get("state_mean", 0.0)),
        "state_std": _scalar(raw["state_std"]),
        "delta_std": _scalar(raw["delta_std"]),
        "source": str(path),
    }


@torch.no_grad()
def estimate_carrier_std(task_name, max_frames=24):
    spec = TASK_SPECS[task_name]
    store = STORES[task_name]["train"]
    pairs = [(trajectory, frame)
             for trajectory in range(store.n_trajectories)
             for frame in range(store.n_frames)]
    ids = np.linspace(0, len(pairs) - 1, min(max_frames, len(pairs)), dtype=np.int64)
    sumsq = torch.zeros(3, device=device, dtype=torch.float64)
    count = 0
    for index in ids:
        trajectory, frame = pairs[int(index)]
        state = torch.from_numpy(store.read_frame(trajectory, frame)[None].copy()).to(device)
        carrier = raw_carrier_3d(state, spec).double()
        sumsq += carrier.square().sum((0, 2, 3, 4))
        count += carrier.shape[0] * carrier.shape[-3] * carrier.shape[-2] * carrier.shape[-1]
    return torch.sqrt(sumsq / max(count, 1)).clamp_min(1e-7).cpu().tolist()


STATS = {}
for name in TASKS_TO_RUN:
    stats = load_base_stats(name)
    stats["carrier_std"] = estimate_carrier_std(name)
    STATS[name] = stats
    print(name, json.dumps(stats, indent=2))

## 5. The one shared trainable model: `GRNO3DCore`

The model is the rank-matched 3D analogue of GRNO-2D: one feature pyramid, one
route pyramid, direct centered--routed concatenation at all four scales, and one
learned residual. Axial depthwise kernels keep the 64-cubed run feasible on one
A100 while preserving the same encoder--decoder organization.

In [ ]:
def pad_axis_3d(x, axis, pad, boundary):
    if pad == 0:
        return x
    pads = {
        -1: (pad, pad, 0, 0, 0, 0),
        -2: (0, 0, pad, pad, 0, 0),
        -3: (0, 0, 0, 0, pad, pad),
    }[axis]
    return F.pad(x, pads, mode="circular" if boundary == "periodic" else "replicate")


class ChannelLayerNorm3D(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x):
        y = x.permute(0, 2, 3, 4, 1)
        y = F.layer_norm(y, (x.shape[1],), self.weight, self.bias, self.eps)
        return y.permute(0, 4, 1, 2, 3)


class BoundaryAxialConvNeXt3D(nn.Module):
    def __init__(self, channels, expansion=2):
        super().__init__()
        self.dw_z = nn.Conv3d(channels, channels, (5, 1, 1), groups=channels)
        self.dw_y = nn.Conv3d(channels, channels, (1, 5, 1), groups=channels)
        self.dw_x = nn.Conv3d(channels, channels, (1, 1, 5), groups=channels)
        self.norm = ChannelLayerNorm3D(channels)
        self.pw1 = nn.Conv3d(channels, expansion * channels, 1)
        self.pw2 = nn.Conv3d(expansion * channels, channels, 1)

    def forward(self, x, boundary_zyx):
        bz, by, bx = boundary_zyx
        z = self.dw_z(pad_axis_3d(x, -3, 2, bz))
        y = self.dw_y(pad_axis_3d(x, -2, 2, by))
        q = self.dw_x(pad_axis_3d(x, -1, 2, bx))
        h = (z + y + q) / 3.0
        return x + self.pw2(F.gelu(self.pw1(self.norm(h))))


class Stage3D(nn.Module):
    def __init__(self, channels, depth):
        super().__init__()
        self.blocks = nn.ModuleList([
            BoundaryAxialConvNeXt3D(channels) for _ in range(depth)
        ])

    def forward(self, x, boundary_zyx):
        for block in self.blocks:
            if self.training and x.is_cuda and torch.is_grad_enabled():
                x = checkpoint(lambda value, b=block: b(value, boundary_zyx),
                               x, use_reentrant=False)
            else:
                x = block(x, boundary_zyx)
        return x


class GRNO3DCore(nn.Module):
    """Task-agnostic trainable graph shared by both 3D PDEs."""
    def __init__(self, width=24, depths=(2, 2, 3, 4)):
        super().__init__()
        if len(depths) != 4:
            raise ValueError("GRNO3DCore requires four stages")
        dims = (width, 2 * width, 3 * width, 4 * width)
        self.dims = dims
        # q, carrier-x/y/z, context, requested dt.
        self.stem = nn.Conv3d(6, dims[0], 1)
        self.enc0 = Stage3D(dims[0], depths[0])
        self.down01 = nn.Conv3d(dims[0], dims[1], 1)
        self.enc1 = Stage3D(dims[1], depths[1])
        self.down12 = nn.Conv3d(dims[1], dims[2], 1)
        self.enc2 = Stage3D(dims[2], depths[2])
        self.down23 = nn.Conv3d(dims[2], dims[3], 1)
        self.enc3 = Stage3D(dims[3], depths[3])

        self.fuse3 = nn.Conv3d(2 * dims[3], dims[3], 1)
        self.dec3 = Stage3D(dims[3], 1)
        self.up32 = nn.Conv3d(dims[3], dims[2], 1)
        self.fuse2 = nn.Conv3d(3 * dims[2], dims[2], 1)
        self.dec2 = Stage3D(dims[2], max(1, depths[2] // 2))
        self.up21 = nn.Conv3d(dims[2], dims[1], 1)
        self.fuse1 = nn.Conv3d(3 * dims[1], dims[1], 1)
        self.dec1 = Stage3D(dims[1], max(1, depths[1] // 2))
        self.up10 = nn.Conv3d(dims[1], dims[0], 1)
        self.fuse0 = nn.Conv3d(3 * dims[0], dims[0], 1)
        self.dec0 = Stage3D(dims[0], max(1, depths[0] // 2))
        self.head = nn.Sequential(
            ChannelLayerNorm3D(dims[0]),
            nn.Conv3d(dims[0], dims[0], 1),
            nn.GELU(),
            nn.Conv3d(dims[0], 1, 1),
        )
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    @staticmethod
    def _pool(x):
        return F.avg_pool3d(x, 2, 2)

    def forward(self, features, departure, lengths_zyx, boundary_zyx):
        if features.shape[1] != 6 or departure.shape[1] != 3:
            raise ValueError((features.shape, departure.shape))
        s0 = self.enc0(self.stem(features), boundary_zyx)
        d1 = self._pool(departure)
        s1 = self.enc1(self.down01(self._pool(s0)), boundary_zyx)
        d2 = self._pool(d1)
        s2 = self.enc2(self.down12(self._pool(s1)), boundary_zyx)
        d3 = self._pool(d2)
        s3 = self.enc3(self.down23(self._pool(s2)), boundary_zyx)

        r3 = boundary_warp3d(s3, d3, lengths_zyx, boundary_zyx)
        z3 = self.dec3(self.fuse3(torch.cat((s3, r3), dim=1)), boundary_zyx)
        up2 = F.interpolate(self.up32(z3), size=s2.shape[-3:], mode="nearest")
        r2 = boundary_warp3d(s2, d2, lengths_zyx, boundary_zyx)
        z2 = self.dec2(self.fuse2(torch.cat((up2, s2, r2), dim=1)), boundary_zyx)
        up1 = F.interpolate(self.up21(z2), size=s1.shape[-3:], mode="nearest")
        r1 = boundary_warp3d(s1, d1, lengths_zyx, boundary_zyx)
        z1 = self.dec1(self.fuse1(torch.cat((up1, s1, r1), dim=1)), boundary_zyx)
        up0 = F.interpolate(self.up10(z1), size=s0.shape[-3:], mode="nearest")
        r0 = boundary_warp3d(s0, departure, lengths_zyx, boundary_zyx)
        z0 = self.dec0(self.fuse0(torch.cat((up0, s0, r0), dim=1)), boundary_zyx)
        return self.head(z0)


def build_model():
    return GRNO3DCore(RUN.width, RUN.depths)


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def trainable_schema(model):
    return [(name, tuple(parameter.shape), str(parameter.dtype))
            for name, parameter in model.named_parameters()]


reference = build_model()
REFERENCE_SCHEMA = trainable_schema(reference)
CORE_SCHEMA_HASH = short_hash(REFERENCE_SCHEMA)
CORE_PARAMETER_COUNT = parameter_count(reference)
audit = []
for task_name in TASKS_TO_RUN:
    candidate = build_model()
    audit.append({
        "task": task_name, "class": candidate.__class__.__name__,
        "parameters": parameter_count(candidate),
        "schema_hash": short_hash(trainable_schema(candidate)),
    })
    assert trainable_schema(candidate) == REFERENCE_SCHEMA
    assert parameter_count(candidate) == CORE_PARAMETER_COUNT
ARCHITECTURE_AUDIT = pd.DataFrame(audit)
print(ARCHITECTURE_AUDIT.to_string(index=False))
print("SHARED GRNO-3D CORE:", CORE_SCHEMA_HASH, CORE_PARAMETER_COUNT)

## 6. Fixed task adapters and model preflight

In [ ]:
def make_features_3d(state, condition, carrier, spec, stats, dt):
    state_mean = float(stats["state_mean"])
    state_std = max(float(stats["state_std"]), 1e-7)
    carrier_scale = torch.as_tensor(
        stats["carrier_std"], device=state.device, dtype=state.dtype
    )[None, :, None, None, None].clamp_min(1e-7)
    dt_channel = torch.full_like(state, float(dt) / spec.frame_dt)
    return torch.cat((
        (state - state_mean) / state_std,
        carrier / carrier_scale,
        condition,
        dt_channel,
    ), dim=1)


def task_step_3d(model, state, condition, spec, stats, dt=None,
                 route_mode="correct", return_aux=False):
    dt = spec.frame_dt if dt is None else float(dt)
    with torch.autocast(device_type=state.device.type, enabled=False):
        carrier = raw_carrier_3d(state.float(), spec)
        departure = midpoint_departure_3d(carrier, dt, spec)
        departure = intervene_route_3d(departure, route_mode)
    features = make_features_3d(state, condition, carrier, spec, stats, dt)
    raw_delta = model(features, departure, spec.lengths_zyx, spec.boundary_zyx)
    delta = raw_delta.float() * float(stats["delta_std"])
    if spec.conserve_mean:
        delta = delta - delta.mean((-3, -2, -1), keepdim=True)
    prediction = state.float() + delta
    if return_aux:
        return prediction, {"carrier": carrier, "departure": departure,
                            "delta": delta}
    return prediction


def preflight_one_task_3d(task_name):
    spec, stats = TASK_SPECS[task_name], STATS[task_name]
    store = STORES[task_name]["train"]
    state = torch.from_numpy(store.read_frame(0, 0)[None].copy()).to(device)
    condition = torch.from_numpy(store.read_condition(0)[None].copy()).to(device)
    model = build_model().to(device)
    model.eval()
    with torch.no_grad():
        prediction, aux = task_step_3d(model, state, condition, spec, stats,
                                       return_aux=True)
    identity = float(torch.linalg.vector_norm(prediction - state) /
                     torch.linalg.vector_norm(state).clamp_min(1e-8))
    route_rms = float(aux["departure"].square().mean().sqrt())
    with torch.no_grad():
        model.head[-1].weight.normal_(std=1e-3)
        model.head[-1].bias.zero_()
    model.train()
    model.zero_grad(set_to_none=True)
    prediction = task_step_3d(model, state, condition, spec, stats)
    loss = (prediction - state).square().mean()
    loss.backward()
    grad_norm = math.sqrt(sum(
        float(parameter.grad.detach().float().square().sum())
        for parameter in model.parameters() if parameter.grad is not None))
    result = {
        "task": task_name, "parameters": parameter_count(model),
        "schema_hash": short_hash(trainable_schema(model)),
        "zero_head_identity_error": identity,
        "route_rms_physical": route_rms,
        "full_graph_gradient_norm": grad_norm,
        "peak_allocated_GB": torch.cuda.max_memory_allocated() / 1e9
            if torch.cuda.is_available() else 0.0,
    }
    print(json.dumps(result, indent=2))
    assert identity < 2e-6 and route_rms > 0 and grad_norm > 0
    del model, state, condition, prediction, loss, aux
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


PREFLIGHT = pd.DataFrame([preflight_one_task_3d(name) for name in TASKS_TO_RUN])
print(PREFLIGHT.to_string(index=False))

## 7. Deterministic windows, common loss, and metrics

In [ ]:
def relative_l2(pred, target, eps=1e-8):
    return (torch.linalg.vector_norm((pred - target).flatten(1), dim=1) /
            torch.linalg.vector_norm(target.flatten(1), dim=1).clamp_min(eps))


def axis_derivative_3d(value, axis, length, boundary):
    n = value.shape[axis]
    spacing = float(length) / n
    if boundary == "periodic":
        return (torch.roll(value, -1, axis) - torch.roll(value, 1, axis)) / (2 * spacing)
    out = torch.zeros_like(value)
    sl = [slice(None)] * value.ndim
    mid, hi, lo = sl.copy(), sl.copy(), sl.copy()
    mid[axis], hi[axis], lo[axis] = slice(1, -1), slice(2, None), slice(None, -2)
    out[tuple(mid)] = (value[tuple(hi)] - value[tuple(lo)]) / (2 * spacing)
    first, second = sl.copy(), sl.copy()
    first[axis], second[axis] = 0, 1
    last, before = sl.copy(), sl.copy()
    last[axis], before[axis] = -1, -2
    out[tuple(first)] = (value[tuple(second)] - value[tuple(first)]) / spacing
    out[tuple(last)] = (value[tuple(last)] - value[tuple(before)]) / spacing
    return out


def spectral_relative_3d(pred, target, spec, eps=1e-8):
    dims = [axis for axis, boundary in zip((-3, -2, -1), spec.boundary_zyx)
            if boundary == "periodic"]
    if not dims:
        return pred.new_tensor(0.0)
    pa = torch.fft.fftn(pred.float(), dim=tuple(dims)).abs()
    ta = torch.fft.fftn(target.float(), dim=tuple(dims)).abs()
    num = (pa - ta).square().flatten(1).sum(1)
    den = ta.square().flatten(1).sum(1).clamp_min(eps)
    return (num / den).mean()


def task_loss_3d(pred, target, spec, stats):
    scale = max(float(stats["state_std"]), 1e-7)
    mse = ((pred - target) / scale).square().mean()
    grad_num = pred.new_tensor(0.0)
    grad_den = pred.new_tensor(0.0)
    for axis, length, boundary in zip((-3, -2, -1), spec.lengths_zyx,
                                      spec.boundary_zyx):
        pg = axis_derivative_3d(pred, axis, length, boundary)
        tg = axis_derivative_3d(target, axis, length, boundary)
        grad_num += (pg - tg).square().mean()
        grad_den += tg.square().mean()
    grad = grad_num / grad_den.clamp_min(1e-8)
    spectral = (spectral_relative_3d(pred, target, spec)
                if RUN.spectral_weight else pred.new_tensor(0.0))
    total = mse + RUN.grad_weight * grad + RUN.spectral_weight * spectral
    return total, {"mse": mse.detach(), "grad": grad.detach(),
                   "spectrum": spectral.detach()}


def relative_h1_3d(pred, target, spec, eps=1e-8):
    num = (pred - target).square().flatten(1).sum(1)
    den = target.square().flatten(1).sum(1)
    for axis, length, boundary in zip((-3, -2, -1), spec.lengths_zyx,
                                      spec.boundary_zyx):
        pg = axis_derivative_3d(pred, axis, length, boundary)
        tg = axis_derivative_3d(target, axis, length, boundary)
        num += (pg - tg).square().flatten(1).sum(1)
        den += tg.square().flatten(1).sum(1)
    return torch.sqrt(num / den.clamp_min(eps))


def spatial_correlation(pred, target, eps=1e-8):
    p, t = pred.flatten(1), target.flatten(1)
    p, t = p - p.mean(1, keepdim=True), t - t.mean(1, keepdim=True)
    return (p * t).sum(1) / (
        torch.linalg.vector_norm(p, dim=1) * torch.linalg.vector_norm(t, dim=1)
    ).clamp_min(eps)


def manifest_path_3d(task_name, seed):
    path = MANIFEST_ROOT / task_name
    path.mkdir(parents=True, exist_ok=True)
    signature = short_hash({
        "task": task_name, "epochs": RUN.epochs,
        "samples": RUN.samples_per_epoch, "K": RUN.train_rollout,
        "data_contract": DATA_CONTRACT_HASHES[task_name], "seed": int(seed),
    })
    return path / f"windows_seed_{seed}_{signature}.npz"


def load_or_create_manifest_3d(task_name, seed):
    store = STORES[task_name]["train"]
    path = manifest_path_3d(task_name, seed)
    expected = (RUN.epochs, RUN.samples_per_epoch, 2)
    if path.exists():
        references = np.load(path)["references"]
        if references.shape != expected:
            raise RuntimeError((path, references.shape, expected))
        return references
    rng_seed = (1_000_003 + int(seed)
                if task_name == "keller_segel3d" else int(seed) + 9_173)
    rng = np.random.default_rng(rng_seed)
    candidates = np.asarray([
        (trajectory, start)
        for trajectory in range(store.n_trajectories)
        for start in range(store.max_train_start + 1)
    ], dtype=np.int64)
    chosen = rng.integers(len(candidates), size=expected[:2])
    references = candidates[chosen]
    tmp = path.with_suffix(".partial.npz")
    np.savez_compressed(tmp, references=references)
    os.replace(tmp, path)
    return references


class ManifestDataset3D(Dataset):
    def __init__(self, store, references, seed):
        self.store, self.references, self.seed = store, references, int(seed)
        self.epoch = 0
        self.order = np.arange(references.shape[1])

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        rng = np.random.default_rng(50_021 + self.seed + 1_000_003 * self.epoch)
        self.order = rng.permutation(self.references.shape[1])

    def __len__(self):
        return self.references.shape[1]

    def __getitem__(self, index):
        trajectory, start = self.references[self.epoch, int(self.order[index])]
        sequence = self.store.read_window(trajectory, start, RUN.train_rollout)
        condition = self.store.read_condition(trajectory)
        return torch.from_numpy(sequence.copy()), torch.from_numpy(condition.copy())


class FixedWindowDataset3D(Dataset):
    def __init__(self, store, horizon, split):
        self.store, self.horizon = store, int(horizon)
        max_start = store.n_frames - self.horizon - 1
        if split == "test":
            if store.task_name == "porous_convection3d":
                # Preserve the original Darcy fixed-start protocol.
                stride = max(1, max_start // 3)
                candidates = (0, stride, 2 * stride, 3 * stride)
                self.references = [
                    (trajectory, min(candidates[trajectory % 4], max_start))
                    for trajectory in range(store.n_trajectories)
                ]
            else:
                starts = np.linspace(0, max_start, store.n_trajectories, dtype=np.int64)
                self.references = list(enumerate(starts.tolist()))
        else:
            candidates = [(trajectory, start)
                          for trajectory in range(store.n_trajectories)
                          for start in range(max_start + 1)]
            ids = np.linspace(0, len(candidates) - 1,
                              min(32, len(candidates)), dtype=np.int64)
            self.references = [candidates[int(index)] for index in ids]

    def __len__(self):
        return len(self.references)

    def __getitem__(self, index):
        trajectory, start = self.references[index]
        return (torch.from_numpy(self.store.read_window(
                    trajectory, start, self.horizon).copy()),
                torch.from_numpy(self.store.read_condition(trajectory).copy()),
                int(trajectory), int(start))

## 8. Training

In [ ]:
def task_signature_3d(task_name, seed):
    return short_hash({
        "model_version": MODEL_VERSION, "core_schema": CORE_SCHEMA_HASH,
        "run": asdict(RUN), "task": asdict(TASK_SPECS[task_name]),
        "stats": STATS[task_name],
        "data": {split: STORES[task_name][split].schema()
                 for split in ("train", "val", "test")},
        "data_contract": DATA_CONTRACT_HASHES[task_name], "seed": int(seed),
    })


def checkpoint_dir_3d(task_name, seed):
    path = OUTPUT_ROOT / task_name / f"seed_{seed}" / task_signature_3d(task_name, seed)
    path.mkdir(parents=True, exist_ok=True)
    return path


@torch.no_grad()
def validate_model_3d(model, task_name):
    spec, stats = TASK_SPECS[task_name], STATS[task_name]
    dataset = FixedWindowDataset3D(STORES[task_name]["val"], RUN.val_rollout, "val")
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0,
                        pin_memory=(device.type == "cuda"))
    model.eval()
    all_steps, finals = [], []
    for sequence, condition, _, _ in loader:
        sequence = sequence.to(device, non_blocking=True)
        condition = condition.to(device, non_blocking=True)
        state = sequence[:, 0]
        values = []
        for step in range(RUN.val_rollout):
            with amp_context(RUN.use_amp):
                state = task_step_3d(model, state, condition, spec, stats)
            values = relative_l2(state.float(), sequence[:, step + 1]).cpu().tolist()
            all_steps.extend(values)
        finals.extend(values)
    return float(np.mean(all_steps)), float(np.mean(finals))


def train_one_3d(task_name, seed):
    seed_everything(seed)
    spec, stats = TASK_SPECS[task_name], STATS[task_name]
    references = load_or_create_manifest_3d(task_name, seed)
    dataset = ManifestDataset3D(STORES[task_name]["train"], references, seed)
    loader = DataLoader(dataset, batch_size=RUN.micro_batch, shuffle=False,
                        num_workers=0, pin_memory=(device.type == "cuda"))
    model = build_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=RUN.lr,
                                  weight_decay=RUN.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=RUN.epochs, eta_min=0.05 * RUN.lr)
    output = checkpoint_dir_3d(task_name, seed)
    last_path, best_path = output / "last.pt", output / "best.pt"
    signature = task_signature_3d(task_name, seed)
    start_epoch, best, history = 0, float("inf"), []
    if RESUME_IF_AVAILABLE and last_path.exists():
        payload = safe_torch_load(last_path)
        if payload.get("signature") != signature:
            raise RuntimeError(f"Checkpoint signature mismatch: {last_path}")
        model.load_state_dict(payload["model"])
        optimizer.load_state_dict(payload["optimizer"])
        scheduler.load_state_dict(payload["scheduler"])
        if "rng_state" in payload:
            restore_rng_state(payload["rng_state"])
        start_epoch = int(payload["epoch"]) + 1
        best = float(payload["best"])
        history = list(payload.get("history", []))
        print(f"Resuming {task_name} seed {seed} at epoch {start_epoch}")

    weights = [RUN.rollout_decay ** step for step in range(RUN.train_rollout)]
    for epoch in range(start_epoch, RUN.epochs):
        dataset.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sums = {key: 0.0 for key in ("loss", "mse", "grad", "spectrum")}
        updates = 0
        started = time.time()
        progress = tqdm(loader, desc=f"GRNO3D {task_name} s{seed} e{epoch+1}/{RUN.epochs}")
        for micro_step, (sequence, condition) in enumerate(progress):
            sequence = sequence.to(device, non_blocking=True)
            condition = condition.to(device, non_blocking=True)
            state = sequence[:, 0]
            with amp_context(RUN.use_amp):
                total = 0.0
                components = {key: state.new_tensor(0.0)
                              for key in ("mse", "grad", "spectrum")}
                for step in range(RUN.train_rollout):
                    state = task_step_3d(model, state, condition, spec, stats)
                    one, parts = task_loss_3d(state, sequence[:, step + 1], spec, stats)
                    total = total + weights[step] * one
                    for key in components:
                        components[key] += parts[key]
                loss = total / sum(weights)
            if not torch.isfinite(loss):
                raise FloatingPointError((task_name, seed, epoch, micro_step, float(loss)))
            (loss / RUN.grad_accum).backward()
            sums["loss"] += float(loss.detach())
            for key in components:
                sums[key] += float(components[key] / RUN.train_rollout)
            if (micro_step + 1) % RUN.grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                updates += 1
            if (micro_step + 1) % 20 == 0:
                progress.set_postfix(loss=f"{sums['loss']/(micro_step+1):.3e}",
                                     updates=updates)
        if updates != RUN.updates_per_epoch:
            raise RuntimeError((updates, RUN.updates_per_epoch))
        scheduler.step()
        val_mean, val_final = validate_model_3d(model, task_name)
        row = {
            "epoch": epoch + 1,
            "train_loss": sums["loss"] / len(loader),
            "train_mse": sums["mse"] / len(loader),
            "train_grad": sums["grad"] / len(loader),
            "train_spectrum": sums["spectrum"] / len(loader),
            "val_relative_l2_mean": val_mean,
            "val_relative_l2_final": val_final,
            "selection_score": val_mean,
            "optimizer_updates": updates,
            "lr": optimizer.param_groups[0]["lr"],
            "seconds": time.time() - started,
        }
        history.append(row)
        print(json.dumps(row, indent=2))
        payload = {
            "model": model.state_dict(), "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "epoch": epoch,
            "best": min(best, val_mean), "history": history,
            "signature": signature, "task": asdict(spec), "stats": stats,
            "run": asdict(RUN), "core_schema_hash": CORE_SCHEMA_HASH,
            "parameters": parameter_count(model), "rng_state": capture_rng_state(),
        }
        atomic_torch_save(last_path, payload)
        if val_mean < best:
            best = val_mean
            payload["best"] = best
            atomic_torch_save(best_path, payload)
        pd.DataFrame(history).to_csv(output / "history.csv", index=False)

    del model, optimizer, scheduler, dataset, loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_path


BEST_PATHS = {}
for task_name in TASKS_TO_RUN:
    for seed in SEEDS:
        if RUN_TRAINING:
            BEST_PATHS[(task_name, seed)] = train_one_3d(task_name, seed)
        else:
            path = checkpoint_dir_3d(task_name, seed) / "best.pt"
            if not path.exists():
                raise FileNotFoundError(path)
            BEST_PATHS[(task_name, seed)] = path
print({str(key): str(path) for key, path in BEST_PATHS.items()})

## 9. H-step free-rollout evaluation and route interventions

In [ ]:
ROUTE_MODES = ("correct", "zero", "reverse", "shuffle")


@torch.no_grad()
def evaluate_one_3d(task_name, seed, route_mode="correct"):
    spec, stats = TASK_SPECS[task_name], STATS[task_name]
    model = build_model().to(device)
    payload = safe_torch_load(BEST_PATHS[(task_name, seed)])
    if payload.get("core_schema_hash") != CORE_SCHEMA_HASH:
        raise RuntimeError("Checkpoint does not belong to the shared GRNO-3D core")
    model.load_state_dict(payload["model"])
    model.eval()
    dataset = FixedWindowDataset3D(STORES[task_name]["test"], RUN.test_rollout, "test")
    rows, sample = [], None
    for item in tqdm(range(len(dataset)), desc=f"test {task_name} {route_mode} s{seed}"):
        sequence, condition, trajectory, start = dataset[item]
        sequence = sequence[None].to(device)
        condition = condition[None].to(device)
        state = sequence[:, 0]
        initial = state.clone()
        truth_slices = [initial[0, 0, initial.shape[-3] // 2].cpu()]
        prediction_slices = [truth_slices[0]]
        for step in range(1, RUN.test_rollout + 1):
            with amp_context(RUN.use_amp):
                state = task_step_3d(
                    model, state, condition, spec, stats, route_mode=route_mode)
            target = sequence[:, step].float()
            prediction = state.float()
            l2 = relative_l2(prediction, target).item()
            persistence = relative_l2(initial.float(), target).item()
            row = {
                "model": "SharedGRNO3D", "task": task_name, "seed": seed,
                "route_mode": route_mode, "trajectory": trajectory,
                "window_start": start, "step": step,
                "relative_l2": l2,
                "relative_h1": relative_h1_3d(prediction, target, spec).item(),
                "correlation": spatial_correlation(prediction, target).item(),
                "spectral_error": float(torch.sqrt(
                    spectral_relative_3d(prediction, target, spec).clamp_min(0))),
                "persistence_relative_l2": persistence,
                "skill_vs_persistence": 1 - l2 / max(persistence, 1e-8),
                "mean_drift": float((prediction.mean() - initial.mean()).abs()),
                "negative_fraction": float((prediction < 0).float().mean()),
            }
            rows.append(row)
            if item == 0:
                center = prediction.shape[-3] // 2
                truth_slices.append(target[0, 0, center].cpu())
                prediction_slices.append(prediction[0, 0, center].cpu())
        if item == 0:
            sample = {"truth": torch.stack(truth_slices),
                      "prediction": torch.stack(prediction_slices)}
    del model, dataset
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame(rows), sample


TEST_FRAMES, SAMPLE_ROLLOUTS = [], {}
if RUN_TESTING:
    for task_name in TASKS_TO_RUN:
        modes = ROUTE_MODES if RUN_ROUTE_INTERVENTIONS else ("correct",)
        for seed in SEEDS:
            for mode in modes:
                frame, sample = evaluate_one_3d(task_name, seed, mode)
                TEST_FRAMES.append(frame)
                if mode == "correct":
                    SAMPLE_ROLLOUTS[(task_name, seed)] = sample

TEST_RESULTS = pd.concat(TEST_FRAMES, ignore_index=True) if TEST_FRAMES else pd.DataFrame()
if not TEST_RESULTS.empty:
    TEST_RESULTS.to_csv(OUTPUT_ROOT / "all_step_results.csv", index=False)
    FINAL = TEST_RESULTS[TEST_RESULTS.step == RUN.test_rollout].copy()
    numeric = [column for column in FINAL.columns
               if column not in {"seed", "trajectory", "window_start", "step"}
               and pd.api.types.is_numeric_dtype(FINAL[column])]
    FINAL_PER_SEED = FINAL.groupby(
        ["task", "route_mode", "seed"], as_index=False)[numeric].mean()
    FINAL_AGGREGATE = FINAL_PER_SEED.groupby(
        ["task", "route_mode"])[numeric].agg(["mean", "std"]).reset_index()
    FINAL_PER_SEED.to_csv(OUTPUT_ROOT / "final_per_seed.csv", index=False)
    FINAL_AGGREGATE.to_csv(OUTPUT_ROOT / "final_aggregate_across_seeds.csv", index=False)
    display(FINAL_AGGREGATE)

## 10. Compact plots and acceptance gate

Both tasks must show the same architecture hash and parameter count. The first
scientific check is whether each shared-core seed remains close to its original
single-task GRNO result while retaining an advantage over the corresponding
baselines.

In [ ]:
if not TEST_RESULTS.empty:
    correct = TEST_RESULTS[TEST_RESULTS.route_mode == "correct"]
    curves = correct.groupby(["task", "step"], as_index=False).relative_l2.mean()
    for task_name in curves.task.unique():
        frame = curves[curves.task == task_name]
        plt.figure(figsize=(5.5, 3.8))
        plt.plot(frame.step, frame.relative_l2)
        plt.xlabel("rollout step")
        plt.ylabel("relative L2")
        plt.title(task_name)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.show()

    interventions = TEST_RESULTS[TEST_RESULTS.step == RUN.test_rollout].groupby(
        ["task", "route_mode"], as_index=False).relative_l2.mean()
    display(interventions)